### RAG Pipelines - Data Ingestion to Vector DB Pipeline


In [8]:
import os
from langchain_community.document_loaders import PyPDFLoader,PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

In [10]:
### Read all the pdf's inside the directory

def process_all_pdfs(pdf_directory):
    """ Process all the PDF files in a directory"""

    all_documents =[]
    pdf_dir = Path(pdf_directory)

    # Find all the pdf files in the directory recursively and return as a list all this done by .rglob() which searches file recursively with the file endign .pdf
    pdf_files = list(pdf_dir.rglob("*.pdf"))

    print(f"Found {len(pdf_files)} PDF files in {pdf_directory}")

    for pdf_file in pdf_files:
        print(f"\nProcessing {pdf_file.name}...")
        try:
            # Load the PDF file using PyMuPDFLoader
            loader = PyMuPDFLoader(str(pdf_file))
            documents = loader.load()

            # Add source infomation to metadata
            for doc in documents:
                doc.metadata["source_file"] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'

            all_documents.append(documents)
            print(f"Loaded {len(documents)} pages from {pdf_file.name}")
                
        except Exception as e:
            print(f"Error processing {pdf_file.name}: {e}")
        
    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents

all_pdf_documents = process_all_pdfs("../data/pdf")

Found 3 PDF files in ../data/pdf

Processing FINALEPICS_finalreport_1-1.pdf...
Loaded 27 pages from FINALEPICS_finalreport_1-1.pdf

Processing Facto_JD_Founders_Office_Sales_Growth.pdf...
Loaded 3 pages from Facto_JD_Founders_Office_Sales_Growth.pdf

Processing HCSTGAI01052026020.pdf...
Loaded 1 pages from HCSTGAI01052026020.pdf

Total documents loaded: 3


In [11]:
all_pdf_documents

[[Document(metadata={'producer': 'Microsoft® Word 2021', 'creator': 'Microsoft® Word 2021', 'creationdate': '2026-03-31T18:42:33+05:30', 'source': '../data/pdf/FINALEPICS_finalreport_1-1.pdf', 'file_path': '../data/pdf/FINALEPICS_finalreport_1-1.pdf', 'total_pages': 27, 'format': 'PDF 1.7', 'title': '', 'author': 'Un-named', 'subject': '', 'keywords': '', 'moddate': '2026-03-31T18:42:33+05:30', 'trapped': '', 'modDate': "D:20260331184233+05'30'", 'creationDate': "D:20260331184233+05'30'", 'page': 0, 'source_file': 'FINALEPICS_finalreport_1-1.pdf', 'file_type': 'pdf'}, page_content='1 \n \n \nAI-Driven Learning Support System with Content Curation and Smart \nLearning \n \n \nAn Engineering Project in Community Service \n \n \nPhase 2 - Final Report \n \nSubmitted by \nSl. No.       Name                                             Register Number \n1.        Prathamesh M Walunj   23BCE10814 \n                                2.        Anuj Kumar Singh           23BCE10738 \n             

In [ ]:
### Text splitting get into chunks

def split_documents(documents, chunk_size=1000, chunk_overlap=200):
    """ Split documents into chunks using RecursiveCharacterTextSplitter"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len
    )
    split_docs =text_splitter.split_documents(documents)  ## It takes the original list of documents and returns a new list conatining smaller Document chunks
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")

    if split_docs:
        print(f"\n Sample Chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")  # Print first 500 characters of the first chunk
        print(f"Metadata: {split_docs[0].metadata}")

    return split_docs

In [21]:
print(type(all_pdf_documents))
print(type(all_pdf_documents[0]))

<class 'list'>
<class 'list'>


In [ ]:
### This converts the nested list into a flat list 
all_pdf_documents = [
    doc
    for pdf_documents in all_pdf_documents
    for doc in pdf_documents
]

In [25]:
chunks = split_documents(all_pdf_documents)
chunks

Split 31 documents into 52 chunks

 Sample Chunk:
Content: 1 
 
 
AI-Driven Learning Support System with Content Curation and Smart 
Learning 
 
 
An Engineering Project in Community Service 
 
 
Phase 2 - Final Report 
 
Submitted by 
Sl. No.       Name     ...
Metadata: {'producer': 'Microsoft® Word 2021', 'creator': 'Microsoft® Word 2021', 'creationdate': '2026-03-31T18:42:33+05:30', 'source': '../data/pdf/FINALEPICS_finalreport_1-1.pdf', 'file_path': '../data/pdf/FINALEPICS_finalreport_1-1.pdf', 'total_pages': 27, 'format': 'PDF 1.7', 'title': '', 'author': 'Un-named', 'subject': '', 'keywords': '', 'moddate': '2026-03-31T18:42:33+05:30', 'trapped': '', 'modDate': "D:20260331184233+05'30'", 'creationDate': "D:20260331184233+05'30'", 'page': 0, 'source_file': 'FINALEPICS_finalreport_1-1.pdf', 'file_type': 'pdf'}


[Document(metadata={'producer': 'Microsoft® Word 2021', 'creator': 'Microsoft® Word 2021', 'creationdate': '2026-03-31T18:42:33+05:30', 'source': '../data/pdf/FINALEPICS_finalreport_1-1.pdf', 'file_path': '../data/pdf/FINALEPICS_finalreport_1-1.pdf', 'total_pages': 27, 'format': 'PDF 1.7', 'title': '', 'author': 'Un-named', 'subject': '', 'keywords': '', 'moddate': '2026-03-31T18:42:33+05:30', 'trapped': '', 'modDate': "D:20260331184233+05'30'", 'creationDate': "D:20260331184233+05'30'", 'page': 0, 'source_file': 'FINALEPICS_finalreport_1-1.pdf', 'file_type': 'pdf'}, page_content='1 \n \n \nAI-Driven Learning Support System with Content Curation and Smart \nLearning \n \n \nAn Engineering Project in Community Service \n \n \nPhase 2 - Final Report \n \nSubmitted by \nSl. No.       Name                                             Register Number \n1.        Prathamesh M Walunj   23BCE10814 \n                                2.        Anuj Kumar Singh           23BCE10738 \n              

### Embedding and VectorStoreDB



In [2]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

/home/oberai/CODE/pytemp/langchain_RAG/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
class EmbeddingManager:
    """ Handles document embedding generation using SentenceTransformer"""

    def __init__(self,model_name:str = "all-MiniLM-L6-v2"):
        """
        Initialize the embedding manager

        Args:
            model_name: Huggingface model name for sentence embeddings.
        """
        self.model_name = model_name
        self.model=None  # self.model is a attribute that will hold the loaded SentenceTransformer model. It is initialized to None, indicating that the model has not been loaded yet.
        self._load_model()
    
    def _load_model(self):
        """Load the sentence transformer model"""
        try:
            print(f"Loading model {self.model_name}...")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model {self.model_name} loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
        Generate embeddings for a list of texts.

        Args:
            texts: List of strings to embed.
        
        Returns:
            Numpy array of embeddings with shape(len(texts)),embedding_dim.
        """
        if not self.model:
            raise ValueError("Model is not loaded. Call load_model() first.")
        
        try:
            print(f"Generating embeddings for {len(texts)} texts...")
            embeddings = self.model.encode(texts, show_progress_bar=True)
            print(f"Generated embeddings with shape: {embeddings.shape}")
            return embeddings
        except Exception as e:
            print(f"Error generating embeddings: {e}")
            raise
    
    def get_sentence_embedding_dimension(self) -> int:
        """Get the dimension of the sentence embeddings."""
        if not self.model:
            raise ValueError("Model is not loaded. Call load_model() first.")
        return self.model.get_sentence_embedding_dimension()


### initialize the embedding manager
embedding_manager = EmbeddingManager()

embedding_manager

Loading model all-MiniLM-L6-v2...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4965.84it/s]


Model all-MiniLM-L6-v2 loaded successfully. Embedding dimension: 384


/tmp/ipykernel_5807/1345579887.py:20: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"Model {self.model_name} loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")
